# **Notebook 1: splitting tiles**

This Jupyter notebook splits the aerial tile from Beeldmateriaal Nederland (2025) into multiple tiles of 1024 x 1024 px.

The script was written for a project commissioned by the Municipality of Wageningen in relation to the course Remote Sensing and GIS integration (GRS60312).

The authors of the script are Polly Cheung, Pascal Dubbelman, Anthony Jansen, Iris Lagemaat, and Susanna van de Wetering.

*Last edited: 15/06/2026*

In [ ]:
from PIL import Image

Image.MAX_IMAGE_PIXELS = None  # Disable the safety limit
im = Image.open(r"C:\Users\antho\Documents\test\2025_173000_443000_RGB_JPEG_hrl.tif")

In [ ]:
import numpy as np
imarray = np.array(im)
print(imarray.shape)
print(im.size)

(20000, 20000, 3)
(20000, 20000)


In [ ]:
import os
import rasterio
from rasterio.windows import Window

# --- SETTINGS ---
tile_size = 1024
overlap = 103

input_path = r"C:\Users\antho\Documents\test\2025_173000_443000_RGB_JPEG_hrl.tif"
output_folder = "tiles"

os.makedirs(output_folder, exist_ok=True)

stride = tile_size - overlap
tile_id = 0

with rasterio.open(input_path) as src:

    for y in range(0, src.height, stride):
        for x in range(0, src.width, stride):

            # Skip incomplete edge tiles
            if x + tile_size > src.width or y + tile_size > src.height:
                continue

            window = Window(x, y, tile_size, tile_size)

            # Read tile
            tile = src.read(window=window)

            # Compute transform for this tile
            transform = src.window_transform(window)

            # Copy metadata
            profile = src.profile.copy()
            profile.update({
                "height": tile_size,
                "width": tile_size,
                "transform": transform
            })

            tile_path = os.path.join(
                output_folder,
                f"tile_{tile_id:04d}.tif"
            )

            with rasterio.open(tile_path, "w", **profile) as dst:
                dst.write(tile)

            tile_id += 1

print(f"Done! Created {tile_id} georeferenced tiles.")


Done! Created 361 georeferenced tiles.


In [ ]:
from PIL import Image

Image.MAX_IMAGE_PIXELS = None  # Disable the safety limit
im = Image.open(r"C:\Users\antho\Documents\test\tiles\tile_0020.tif")

In [ ]:
import numpy as np
imarray = np.array(im)
print(imarray.shape)
print(im.size)

(1024, 1024, 3)
(1024, 1024)
